In [1]:
import pandas as pd
import duckdb
import numpy as np

conn = duckdb.connect()


In [2]:
df = pd.read_parquet("data/faculty-salaries.parquet")


### Origional Data Shape

In [3]:
df.count()

name                5345243
job                 5345131
department              137
base                5345131
overtime            5345131
other               5345131
ual                       0
benefitsee                0
benefitser                0
benefitsdc                0
notes                 28494
totalpay            5345115
totalbenefits       5345131
totalpaybenefits    5345128
year                5345243
dtype: int64

### Data Cleanup
* Remove empty Cols
* Drop missing names and pays
* Base salary must be more than 10k
* Make all strings lowercase

In [4]:
# Drop empty cols: ual, benefitsee, benefitser, and benefitsdc
df = df.drop(columns=['ual', 'benefitsee', 'benefitser', 'benefitsdc', 'notes'])

# Drop rows missing name or totalpay
df = df.dropna(subset=['name', 'totalpay'])
# filter to base salary > 10k
df = df[df['totalpay'] >= 10000]

# make everything lowercase for consistency
df = df.map(lambda s: s.lower() if isinstance(s, str) else s)

df.count()


name                3544483
job                 3544483
department               25
base                3544483
overtime            3544483
other               3544483
totalpay            3544483
totalbenefits       3544483
totalpaybenefits    3544483
year                3544483
dtype: int64

* Since department data is missing for the most part and parcially covered in job we will be removing that as well

In [5]:
df = df.drop(columns=['department'])

### Looking at *Educator* Jobs

In [6]:
duckdb.sql("""
    SELECT 
        job,
        COUNT(*) AS employee_count,
        ROUND(AVG(totalpaybenefits)) AS avg_pay
    FROM df
    WHERE 
        job ILIKE '%prof%'
        OR job ILIKE '%instruct%'
        OR job ILIKE '%faculty%'
        OR job ILIKE '%lectur%'
        OR job ILIKE '%adjunct%'
    GROUP BY job
    ORDER BY employee_count DESC""").df()

,job,employee_count,avg_pay
0,lecturer - academic year,170191,49319.0
1,instructional faculty - academic year,145054,123397.0
2,prof-ay,43728,230251.0
3,hs asst clin prof-hcomp,22291,274080.0
4,assoc prof-ay,18697,151164.0
...,...,...,...
1657,faculty senate president/biology instrctor,1,234720.0
1658,theatre arts instructor,1,205362.0
1659,assos prof bsad,1,182104.0
1660,professor/history,1,269685.0


### Narrow the df to only educational faculty with totalpay >= 10k

In [7]:
df = duckdb.sql("""
    SELECT *
    FROM df
    WHERE 
        job ILIKE '%prof%'
        OR job ILIKE '%instruct%'
        OR job ILIKE '%faculty%'
        OR job ILIKE '%lectur%'
        OR job ILIKE '%adjunct%'
        AND totalpay >= 10000
        """).df()

In [ ]:
duckdb.sql(""" 
    CREATE TABLE faculty_salaries AS
    SELECT *
    FROM df
""")

#### Investigating Faculty with a tenure of at lease 10 years 
First I want just faculty who've been around for 10 years and their pay

In [ ]:
duckdb.sql("""
WITH yearly_pay AS (
    SELECT name, job, year, SUM(totalpay) AS pay
    FROM faculty_salaries
    GROUP BY name, job, year
),
oldtimers AS (
    -- Group for consecutive years
    SELECT name, job, year, pay,
        year - CAST(ROW_NUMBER() OVER (PARTITION BY name ORDER BY year) AS INT) AS grp
    FROM yearly_pay
),
valid_groups AS (
    -- People who have had 10 consecutive years of pay
    SELECT name, grp
    FROM oldtimers
    GROUP BY name, grp
    HAVING COUNT(*) >= 10
)
SELECT oldtimers.name, oldtimers.job, oldtimers.year, ROUND(oldtimers.pay) AS pay
FROM oldtimers
JOIN valid_groups 
    ON oldtimers.name = valid_groups.name 
    AND oldtimers.grp = valid_groups.grp
ORDER BY oldtimers.name, oldtimers.year
""").df()

,name,job,year,pay
0,a dee williams,instructional faculty - academic year,2011,68328.0
1,a dee williams,instructional faculty - academic year,2012,69228.0
2,a dee williams,instructional faculty - academic year,2013,67812.0
3,a dee williams,instructional faculty - academic year,2014,75699.0
4,a dee williams,instructional faculty - academic year,2015,80975.0
...,...,...,...,...
210174,zvonimir hlousek,instructional faculty - academic year,2016,112272.0
210175,zvonimir hlousek,instructional faculty - academic year,2017,101110.0
210176,zvonimir hlousek,instructional faculty - academic year,2018,102258.0
210177,zvonimir hlousek,instructional faculty - academic year,2019,107229.0


* Get faculty who have been around a long time get their starting pay and ending pay

In [22]:
pay_df = duckdb.sql("""
WITH yearly_pay AS (
    SELECT name, year, SUM(totalpay) AS pay,
        arg_max(job, totalpay) AS primary_job
    FROM faculty_salaries
    GROUP BY name, year
),
oldtimers AS (
    SELECT name, year, pay, primary_job,
        year - CAST(ROW_NUMBER() OVER (PARTITION BY name ORDER BY year) AS INT) AS grp
    FROM yearly_pay
),
consecutive_years AS (
    SELECT 
        name,
        MIN(year) AS start_year,
        MAX(year) AS end_year,
        
        -- Primary job
        arg_min(primary_job, year) AS starting_job,
        arg_max(primary_job, year) AS ending_job,
        
        arg_min(pay, year) AS starting_pay,
        arg_max(pay, year) AS ending_pay
    FROM oldtimers
    GROUP BY name, grp
    HAVING COUNT(*) >= 10
)
SELECT 
    name,
    starting_job,
    ending_job,
    start_year,
    end_year,
    starting_pay,
    ending_pay
    FROM consecutive_years
""").df()

#### Median Difference Between Starting and Ending Pay

In [24]:
pay_df['pay_difference'] = pay_df['ending_pay'] - pay_df['starting_pay']

median_diff = pay_df['pay_difference'].median()
mean_diff = pay_df['pay_difference'].mean()

print(f"Median Pay Difference: ${median_diff:,.2f}")
print(f"Mean Pay Difference: ${mean_diff:,.2f}")

Median Pay Difference: $48,723.30
Mean Pay Difference: $56,456.52


**The Median Increase for a faculty who has been working 10 years is ~$48,723.30**

### Overall Pay Stats

In [ ]:
col = df["totalpay"]

col.mean()
col.median()                                       # center — robust
col.std()    
col.min()
col.max()                                        # spread — sensitive
print("Mean:", col.mean())
print("Median:", col.median())
print("Standard Deviation:", col.std())
print("Minimum:", col.min())
print("Maximum:", col.max())
print("IQR:", col.quantile(0.75) - col.quantile(0.25))


Mean: 118664.07659787759
Median: 84135.32
Standard Deviation: 123987.55371484198
Minimum: 10000.0
Maximum: 3974061.0
IQR: 106524.005
